# Macro Pipeline: WDI + PWT + EFW (Paso a Paso)

Notebook organizado por secciones para explicar y auditar cada etapa.
Cada paso guarda dataframes intermedios para revisarlos con Data Wrangler.


In [37]:
from __future__ import annotations
from pathlib import Path
import io
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import requests
import hashlib
import yaml
from datetime import datetime
import wbgapi as wb

pd.options.mode.copy_on_write = True


def find_repo_root(start: Path | None = None) -> Path:
    start = start or Path.cwd()
    for p in [start] + list(start.parents):
        if (p / 'data').exists() and (p / 'notebooks').exists():
            return p
    raise ValueError('Repo root not found')

ROOT = find_repo_root()

PATHS = {
    'data_processed': ROOT / 'data' / 'processed',
    'data_raw': ROOT / 'data' / 'raw',
    'reports': ROOT / 'reports',
}


def read_parquet_pyarrow(path: str | Path) -> pd.DataFrame:
    table = pq.read_table(str(path))
    return table.to_pandas()


def write_parquet_pyarrow(df: pd.DataFrame, path: str | Path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    table = pa.Table.from_pandas(df, preserve_index=False)
    pq.write_table(table, str(path))


REGISTRY_PATH = PATHS['data_raw'] / 'macro' / '_registry.yml'

def read_registry() -> list[dict]:
    if not REGISTRY_PATH.exists():
        return []
    with open(REGISTRY_PATH, 'r') as f:
        data = yaml.safe_load(f) or []
    return data


def write_registry(entries: list[dict]) -> None:
    REGISTRY_PATH.parent.mkdir(parents=True, exist_ok=True)
    with open(REGISTRY_PATH, 'w') as f:
        yaml.safe_dump(entries, f, sort_keys=False)


def update_registry(entry: dict) -> None:
    entries = read_registry()
    entries.append(entry)
    write_registry(entries)


AGGREGATES = set(wb.economy.aggregates())


## Seccion 1: WDI

Primero se define el mapeo de indicadores y la funcion de descarga/caching.


In [38]:
# WDI indicator mapping
INDICATORS = {
    # Real activity / development
    'gdp_pc_real': 'NY.GDP.PCAP.KD',
    'gdp_growth': 'NY.GDP.MKTP.KD.ZG',
    'gdp_pc_growth': 'NY.GDP.PCAP.KD.ZG',
    'inv_gdp': 'NE.GDI.FTOT.ZS',
    'pop': 'SP.POP.TOTL',
    'pop_growth': 'SP.POP.GROW',

    # Prices / monetary
    'inflation_cpi': 'FP.CPI.TOTL.ZG',
    'exrate_lcu_per_usd': 'PA.NUS.FCRF',
    'credit_private_gdp': 'FS.AST.PRVT.GD.ZS',
    'money_broad_gdp': 'FM.LBL.BMNY.GD.ZS',

    # Fiscal / state size
    'gov_cons_gdp': 'NE.CON.GOVT.ZS',
    'tax_rev_gdp': 'GC.TAX.TOTL.GD.ZS',
    'debt_gdp': 'GC.DOD.TOTL.GD.ZS',

    # External sector
    'trade_gdp': 'NE.TRD.GNFS.ZS',
    'exports_gdp': 'NE.EXP.GNFS.ZS',
    'imports_gdp': 'NE.IMP.GNFS.ZS',
    'ca_gdp': 'BN.CAB.XOKA.GD.ZS',
    'natres_rents_gdp': 'NY.GDP.TOTL.RT.ZS',

    # Demography
    'urban_share': 'SP.URB.TOTL.IN.ZS',
    'dep_ratio': 'SP.POP.DPND',
}


In [39]:
# Fetch WDI with caching

def _cache_key(indicators: dict, start: int, end: int) -> str:
    payload = str(sorted(indicators.items())) + f"|{start}|{end}|country_name_v1"
    return hashlib.sha1(payload.encode('utf-8')).hexdigest()[:12]


def fetch_wdi(indicators: dict[str,str], start: int, end: int) -> pd.DataFrame:
    key = _cache_key(indicators, start, end)
    cache_path = PATHS['data_raw'] / 'macro' / 'wdi' / f'cache_{key}.parquet'

    if cache_path.exists():
        print('Using cache:', cache_path)
        return read_parquet_pyarrow(cache_path)

    rows = []
    for var_name, code in indicators.items():
        url = f"https://api.worldbank.org/v2/country/all/indicator/{code}?format=json&per_page=20000&date={start}:{end}"
        resp = requests.get(url, timeout=60)
        resp.raise_for_status()
        data = resp.json()
        if not isinstance(data, list) or len(data) < 2:
            raise ValueError(f"Unexpected response for {code}")

        for item in data[1]:
            iso3 = (item.get('countryiso3code') or '').upper().strip()
            if not iso3:
                continue
            try:
                year = int(item['date'])
            except Exception:
                continue

            country_name = (item.get('country') or {}).get('value')
            val = item.get('value')
            rows.append({
                'iso3': iso3,
                'year': year,
                'country_name': country_name,
                'var_name': var_name,
                'value': val,
                'source': 'wdi',
                'indicator_code': code,
            })
        print('Fetched', var_name, code, 'rows:', len(data[1]))

    df = pd.DataFrame(rows)
    df['iso3'] = df['iso3'].str.upper().str.strip()
    df['country_name'] = df['country_name'].astype('string').str.strip()
    df.loc[df['country_name'] == '', 'country_name'] = pd.NA

    df = df[df['iso3'].str.len() == 3]
    df = df[~df['iso3'].isin(AGGREGATES)]

    write_parquet_pyarrow(df, cache_path)

    # Also write a dated raw file for auditability
    dated = PATHS['data_raw'] / 'macro' / 'wdi' / f"wdi_long_{datetime.now().strftime('%Y%m%d')}.parquet"
    if not dated.exists():
        write_parquet_pyarrow(df, dated)

    # Update registry
    update_registry({
        'dataset': 'wdi',
        'pull_date': datetime.now().isoformat(timespec='seconds'),
        'start_year': start,
        'end_year': end,
        'indicators': indicators,
        'cache_path': str(cache_path),
    })

    return df



## Seccion 1: Carga y transformacion de WDI

Se abre el dataset, se pasa de long a wide y se construye la base `macro_*`.


In [40]:
# Configuracion de exportacion de datasets intermedios/finales
STEP_EXPORT_DIR = PATHS["data_processed"] / "macro_pipeline_step_by_step"
STEP_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
SAVE_CSV = False  # Cambia a True si quieres copias en CSV.

saved_rows = []

# 1) Cargar WDI long
wdi_long_df = fetch_wdi(INDICATORS, start=1970, end=2023).copy()
wdi_long_df["country_name"] = wdi_long_df["country_name"].astype("string").str.strip()
wdi_long_df.loc[wdi_long_df["country_name"] == "", "country_name"] = pd.NA
wdi_long_df["value"] = pd.to_numeric(wdi_long_df["value"], errors="coerce")

wdi_long_path = STEP_EXPORT_DIR / "01_wdi_long.parquet"
write_parquet_pyarrow(wdi_long_df, wdi_long_path)
if SAVE_CSV:
    wdi_long_df.to_csv(STEP_EXPORT_DIR / "01_wdi_long.csv", index=False)

saved_rows.append({
    "dataset": "wdi_long",
    "rows": int(len(wdi_long_df)),
    "cols": int(wdi_long_df.shape[1]),
    "path": str(wdi_long_path),
})
print("[OK] Saved", wdi_long_path)

# 2) Transformar WDI long -> wide y preservar nombre de pais
wdi_country_name_df = (
    wdi_long_df[["iso3", "year", "country_name"]]
    .dropna(subset=["country_name"])
    .drop_duplicates(subset=["iso3", "year"], keep="first")
    .rename(columns={"country_name": "country_name_wdi"})
)

wdi_wide_df = (
    wdi_long_df
    .pivot_table(index=["iso3", "year"], columns="var_name", values="value", aggfunc="mean")
    .reset_index()
    .sort_values(["iso3", "year"])
    .reset_index(drop=True)
)
wdi_wide_df.columns.name = None

wdi_wide_df = wdi_wide_df.merge(
    wdi_country_name_df,
    on=["iso3", "year"],
    how="left",
    validate="one_to_one",
)

wdi_wide_path = STEP_EXPORT_DIR / "02_wdi_wide.parquet"
write_parquet_pyarrow(wdi_wide_df, wdi_wide_path)
if SAVE_CSV:
    wdi_wide_df.to_csv(STEP_EXPORT_DIR / "02_wdi_wide.csv", index=False)

saved_rows.append({
    "dataset": "wdi_wide",
    "rows": int(len(wdi_wide_df)),
    "cols": int(wdi_wide_df.shape[1]),
    "path": str(wdi_wide_path),
})
print("[OK] Saved", wdi_wide_path)

# 3) Renombrar WDI wide a macro_* (sin transformaciones)
wdi_macro_df = wdi_wide_df.copy()
for var in [c for c in wdi_macro_df.columns if c not in ["iso3", "year", "country_name_wdi"]]:
    wdi_macro_df = wdi_macro_df.rename(columns={var: f"macro_{var}"})

front_cols = ["iso3", "year", "country_name_wdi"]
other_cols = [c for c in wdi_macro_df.columns if c not in front_cols]
wdi_macro_df = wdi_macro_df[front_cols + other_cols]

wdi_macro_path = STEP_EXPORT_DIR / "03_wdi_macro_base.parquet"
write_parquet_pyarrow(wdi_macro_df, wdi_macro_path)
if SAVE_CSV:
    wdi_macro_df.to_csv(STEP_EXPORT_DIR / "03_wdi_macro_base.csv", index=False)

saved_rows.append({
    "dataset": "wdi_macro_base",
    "rows": int(len(wdi_macro_df)),
    "cols": int(wdi_macro_df.shape[1]),
    "path": str(wdi_macro_path),
})
print("[OK] Saved", wdi_macro_path)



Using cache: /Users/gabrielsaco/Documents/GitHub/economic-freedom/data/raw/macro/wdi/cache_33e2d7939ac8.parquet
[OK] Saved /Users/gabrielsaco/Documents/GitHub/economic-freedom/data/processed/macro_pipeline_step_by_step/01_wdi_long.parquet
[OK] Saved /Users/gabrielsaco/Documents/GitHub/economic-freedom/data/processed/macro_pipeline_step_by_step/02_wdi_wide.parquet
[OK] Saved /Users/gabrielsaco/Documents/GitHub/economic-freedom/data/processed/macro_pipeline_step_by_step/03_wdi_macro_base.parquet


## Seccion 2: PWT

Ahora se preparan los insumos de PWT desde los CSV locales (sin funciones adicionales).


In [41]:
# Configuracion de insumos PWT (sin funciones)
PWT_CSV_PATHS = [
    ROOT / "pwt70-74.csv",
    ROOT / "pwt75-89.csv",
    ROOT / "pwt90-04.csv",
    ROOT / "pwt2005-23.csv",
]

missing_pwt = [p for p in PWT_CSV_PATHS if not p.exists()]
if missing_pwt:
    raise FileNotFoundError(f"Missing PWT CSV(s): {[str(p) for p in missing_pwt]}")

print("PWT files:", [p.name for p in PWT_CSV_PATHS])



PWT files: ['pwt70-74.csv', 'pwt75-89.csv', 'pwt90-04.csv', 'pwt2005-23.csv']


## Seccion 2: Carga de PWT y merge con WDI

Se abre PWT paso a paso, se limpia y se genera el panel intermedio WDI + PWT.


In [42]:
# 4) Cargar y limpiar PWT desde CSVs (paso a paso)
pwt_frames = []

for path_csv in PWT_CSV_PATHS:
    pwt_part = pd.read_csv(path_csv)

    required_cols = {"iso3", "year", "Country"}
    if not required_cols.issubset(pwt_part.columns):
        raise ValueError(f"Expected columns {required_cols} in {path_csv.name}, got: {list(pwt_part.columns)}")

    pwt_part["iso3"] = pwt_part["iso3"].astype(str).str.upper().str.strip()
    pwt_part["year"] = pd.to_numeric(pwt_part["year"], errors="coerce")
    pwt_part["country_name_pwt"] = pwt_part["Country"].astype("string").str.strip()
    pwt_part.loc[pwt_part["country_name_pwt"] == "", "country_name_pwt"] = pd.NA

    pwt_part = pwt_part[pwt_part["iso3"].str.len() == 3]
    pwt_part = pwt_part[pwt_part["year"].notna()]
    pwt_part["year"] = pwt_part["year"].astype(int)

    value_cols = [c for c in pwt_part.columns if c not in ["iso3", "year", "Country", "country_name_pwt"]]
    numeric_cols = []

    for c in value_cols:
        converted = pd.to_numeric(pwt_part[c], errors="coerce")
        if converted.notna().any():
            pwt_part[c] = converted
            numeric_cols.append(c)

    keep_cols = ["iso3", "year", "country_name_pwt"] + numeric_cols
    pwt_frames.append(pwt_part[keep_cols])

pwt_wide_df = pd.concat(pwt_frames, ignore_index=True)
pwt_wide_df = pwt_wide_df.sort_values(["iso3", "year"]).drop_duplicates(subset=["iso3", "year"], keep="last")

pwt_rename = {
    c: f"pwt_{c}"
    for c in pwt_wide_df.columns
    if c not in ["iso3", "year", "country_name_pwt"]
}
pwt_wide_df = pwt_wide_df.rename(columns=pwt_rename)

front_cols = ["iso3", "year", "country_name_pwt"]
other_cols = [c for c in pwt_wide_df.columns if c not in front_cols]
pwt_wide_df = pwt_wide_df[front_cols + other_cols]

pwt_wide_path = STEP_EXPORT_DIR / "04_pwt_wide.parquet"
write_parquet_pyarrow(pwt_wide_df, pwt_wide_path)
if SAVE_CSV:
    pwt_wide_df.to_csv(STEP_EXPORT_DIR / "04_pwt_wide.csv", index=False)

saved_rows.append({
    "dataset": "pwt_wide",
    "rows": int(len(pwt_wide_df)),
    "cols": int(pwt_wide_df.shape[1]),
    "path": str(pwt_wide_path),
})
print("[OK] Saved", pwt_wide_path)

# 5) Merge WDI + PWT (outer para conservar left_only y right_only)
if wdi_macro_df.duplicated(subset=["iso3", "year"]).any():
    raise ValueError("WDI data has duplicated iso3-year rows")
if pwt_wide_df.duplicated(subset=["iso3", "year"]).any():
    raise ValueError("PWT data has duplicated iso3-year rows")

wdi_keys = wdi_macro_df[["iso3", "year"]]
pwt_keys = pwt_wide_df[["iso3", "year"]]
overlap_wdi_pwt = wdi_keys.merge(pwt_keys, on=["iso3", "year"], how="outer", indicator=True)
print("Merge check (WDI vs PWT):", overlap_wdi_pwt["_merge"].value_counts().to_dict())

wdi_pwt_panel_df = wdi_macro_df.merge(
    pwt_wide_df,
    on=["iso3", "year"],
    how="outer",
    validate="one_to_one",
)
wdi_pwt_panel_df = wdi_pwt_panel_df.sort_values(["iso3", "year"]).reset_index(drop=True)

wdi_pwt_path = STEP_EXPORT_DIR / "05_wdi_pwt_merged.parquet"
write_parquet_pyarrow(wdi_pwt_panel_df, wdi_pwt_path)
if SAVE_CSV:
    wdi_pwt_panel_df.to_csv(STEP_EXPORT_DIR / "05_wdi_pwt_merged.csv", index=False)

saved_rows.append({
    "dataset": "wdi_pwt_merged",
    "rows": int(len(wdi_pwt_panel_df)),
    "cols": int(wdi_pwt_panel_df.shape[1]),
    "path": str(wdi_pwt_path),
})
print("[OK] Saved", wdi_pwt_path)



[OK] Saved /Users/gabrielsaco/Documents/GitHub/economic-freedom/data/processed/macro_pipeline_step_by_step/04_pwt_wide.parquet
Merge check (WDI vs PWT): {'both': 9241, 'left_only': 2477, 'right_only': 162}
[OK] Saved /Users/gabrielsaco/Documents/GitHub/economic-freedom/data/processed/macro_pipeline_step_by_step/05_wdi_pwt_merged.parquet


In [43]:
# ISO3 por grupo del merge WDI vs PWT
merge_groups = ["both", "left_only", "right_only"]

iso3_by_group_wdi_pwt = {
    grp: sorted(
        overlap_wdi_pwt.loc[overlap_wdi_pwt["_merge"] == grp, "iso3"]
        .dropna()
        .unique()
        .tolist()
    )
    for grp in merge_groups
}

iso3_summary_wdi_pwt_df = pd.DataFrame({
    "_merge": merge_groups,
    "n_iso3": [len(iso3_by_group_wdi_pwt[g]) for g in merge_groups],
    "n_rows_iso3_year": [int((overlap_wdi_pwt["_merge"] == g).sum()) for g in merge_groups],
})

iso3_detail_wdi_pwt_df = (
    overlap_wdi_pwt[["_merge", "iso3"]]
    .dropna(subset=["iso3"])
    .drop_duplicates()
    .sort_values(["_merge", "iso3"])
    .reset_index(drop=True)
)

print("Resumen de ISO3 por grupo:")
print(iso3_summary_wdi_pwt_df)

for grp in merge_groups:
    print()
    print(f"{grp} ({len(iso3_by_group_wdi_pwt[grp])} ISO3):")
    print(", ".join(iso3_by_group_wdi_pwt[grp]))

# DataFrames para explorar en Data Wrangler:
# - iso3_summary_wdi_pwt_df
# - iso3_detail_wdi_pwt_df


Resumen de ISO3 por grupo:
       _merge  n_iso3  n_rows_iso3_year
0        both     182              9241
1   left_only      62              2477
2  right_only       3               162

both (182 ISO3):
ABW, AGO, ALB, ARE, ARG, ARM, ATG, AUS, AUT, AZE, BDI, BEL, BEN, BFA, BGD, BGR, BHR, BHS, BIH, BLR, BLZ, BMU, BOL, BRA, BRB, BRN, BTN, BWA, CAF, CAN, CHE, CHL, CHN, CIV, CMR, COD, COG, COL, COM, CPV, CRI, CUW, CYM, CYP, CZE, DEU, DJI, DMA, DNK, DOM, DZA, ECU, EGY, ESP, EST, ETH, FIN, FJI, FRA, GAB, GBR, GEO, GHA, GIN, GMB, GNB, GNQ, GRC, GRD, GTM, GUY, HKG, HND, HRV, HTI, HUN, IDN, IND, IRL, IRN, IRQ, ISL, ISR, ITA, JAM, JOR, JPN, KAZ, KEN, KGZ, KHM, KNA, KOR, KWT, LAO, LBN, LBR, LCA, LKA, LSO, LTU, LUX, LVA, MAC, MAR, MDA, MDG, MDV, MEX, MKD, MLI, MLT, MMR, MNE, MNG, MOZ, MRT, MUS, MWI, MYS, NAM, NER, NGA, NIC, NLD, NOR, NPL, NZL, OMN, PAK, PAN, PER, PHL, POL, PRT, PRY, PSE, QAT, ROU, RUS, RWA, SAU, SDN, SEN, SGP, SLE, SLV, SOM, SRB, SSD, STP, SUR, SVK, SVN, SWE, SWZ, SXM, SYC, SYR, 

## Seccion 3: EFW

Se configura la fuente EFW y luego se procesa en pasos, sin funciones adicionales.


In [44]:
# Configuracion de insumos EFW (sin funciones)
EFW_XLSX_PATH = ROOT / "efw.xlsx"
EFW_SHEET = "EFW Panel Dataset"

if not EFW_XLSX_PATH.exists():
    raise FileNotFoundError(f"EFW Excel not found: {EFW_XLSX_PATH}")

print("EFW file:", EFW_XLSX_PATH.name, "| sheet:", EFW_SHEET)



EFW file: efw.xlsx | sheet: EFW Panel Dataset


In [45]:
# 6) Cargar y limpiar EFW desde Excel (paso a paso)
try:
    efw_raw_df = pd.read_excel(EFW_XLSX_PATH, sheet_name=EFW_SHEET)
except ImportError as e:
    raise ImportError("Reading .xlsx requires openpyxl. Install via `pip install openpyxl`.") from e

print("EFW columns:", list(efw_raw_df.columns))

required_cols_efw = {"ISO_Code", "Countries", "Year"}
if not required_cols_efw.issubset(efw_raw_df.columns):
    raise ValueError(f"Expected columns {required_cols_efw} in EFW sheet. Got: {list(efw_raw_df.columns)}")

efw_wide_df = efw_raw_df.rename(
    columns={
        "ISO_Code": "iso3",
        "Countries": "country_name_efw",
        "Year": "year",
    }
).copy()

efw_wide_df["iso3"] = efw_wide_df["iso3"].astype(str).str.upper().str.strip()
efw_wide_df["year"] = pd.to_numeric(efw_wide_df["year"], errors="coerce")
efw_wide_df["country_name_efw"] = efw_wide_df["country_name_efw"].astype("string").str.strip()
efw_wide_df.loc[efw_wide_df["country_name_efw"] == "", "country_name_efw"] = pd.NA

efw_wide_df = efw_wide_df[efw_wide_df["iso3"].str.len() == 3]
efw_wide_df = efw_wide_df[efw_wide_df["year"].notna()]
efw_wide_df["year"] = efw_wide_df["year"].astype(int)

efwsrc_to_clean = {
    "Summary": "efw_summary",
    "Area 1": "efw_area_1",
    "Area 2": "efw_area_2",
    "Area 3": "efw_area_3",
    "Area 4": "efw_area_4",
    "Area 5": "efw_area_5",
    "Standard Deviation of the 5 EFW Areas": "efw_sd_5_areas",
}

missing_efw_cols = [c for c in efwsrc_to_clean if c not in efw_wide_df.columns]
if missing_efw_cols:
    print("Warning - missing expected EFW columns:", missing_efw_cols)

available_efw_cols = [c for c in efwsrc_to_clean if c in efw_wide_df.columns]
for c in available_efw_cols:
    efw_wide_df[c] = pd.to_numeric(efw_wide_df[c], errors="coerce")

efw_wide_df = efw_wide_df.rename(columns={c: efwsrc_to_clean[c] for c in available_efw_cols})

efwnum_clean_cols = [efwsrc_to_clean[c] for c in available_efw_cols]
keep_cols = ["iso3", "year", "country_name_efw"] + efwnum_clean_cols
efw_wide_df = efw_wide_df[keep_cols].copy()

efw_wide_df = efw_wide_df.drop_duplicates(subset=["iso3", "year"], keep="last")
efw_wide_df = efw_wide_df.sort_values(["iso3", "year"]).reset_index(drop=True)

efw_wide_path = STEP_EXPORT_DIR / "06_efw_wide.parquet"
write_parquet_pyarrow(efw_wide_df, efw_wide_path)
if SAVE_CSV:
    efw_wide_df.to_csv(STEP_EXPORT_DIR / "06_efw_wide.csv", index=False)

saved_rows.append({
    "dataset": "efw_wide",
    "rows": int(len(efw_wide_df)),
    "cols": int(efw_wide_df.shape[1]),
    "path": str(efw_wide_path),
})
print("[OK] Saved", efw_wide_path)

# 7) Merge final (WDI + PWT + EFW)
if efw_wide_df.duplicated(subset=["iso3", "year"]).any():
    raise ValueError("EFW data has duplicated iso3-year rows")

base_keys = wdi_pwt_panel_df[["iso3", "year"]]
efw_keys = efw_wide_df[["iso3", "year"]]
overlap_efw = base_keys.merge(efw_keys, on=["iso3", "year"], how="outer", indicator=True)
print("Merge check (Panel vs EFW):", overlap_efw["_merge"].value_counts().to_dict())

macro_panel_df = wdi_pwt_panel_df.merge(
    efw_wide_df,
    on=["iso3", "year"],
    how="left",
    validate="one_to_one",
)

macro_panel_step_path = STEP_EXPORT_DIR / "07_macro_panel.parquet"
write_parquet_pyarrow(macro_panel_df, macro_panel_step_path)
if SAVE_CSV:
    macro_panel_df.to_csv(STEP_EXPORT_DIR / "07_macro_panel.csv", index=False)

saved_rows.append({
    "dataset": "macro_panel",
    "rows": int(len(macro_panel_df)),
    "cols": int(macro_panel_df.shape[1]),
    "path": str(macro_panel_step_path),
})
print("[OK] Saved", macro_panel_step_path)

# 8) Reportes de cobertura
macro_vars = [c for c in macro_panel_df.columns if c.startswith("macro_")]
cov = pd.DataFrame({
    "var": macro_vars,
    "missing_share_overall": [macro_panel_df[v].isna().mean() for v in macro_vars],
})
write_parquet_pyarrow(cov, PATHS["reports"] / "macro_coverage" / "coverage_overall.parquet")

pwt_vars = [c for c in macro_panel_df.columns if c.startswith("pwt_")]
if pwt_vars:
    cov_pwt = pd.DataFrame({
        "var": pwt_vars,
        "missing_share_overall": [macro_panel_df[v].isna().mean() for v in pwt_vars],
    })
    write_parquet_pyarrow(cov_pwt, PATHS["reports"] / "macro_coverage" / "coverage_pwt_overall.parquet")

efw_vars = [c for c in macro_panel_df.columns if c.startswith("efw_")]
if efw_vars:
    cov_efw = pd.DataFrame({
        "var": efw_vars,
        "missing_share_overall": [macro_panel_df[v].isna().mean() for v in efw_vars],
    })
    write_parquet_pyarrow(cov_efw, PATHS["reports"] / "macro_coverage" / "coverage_efw_overall.parquet")

# 9) Salida final legacy (compatibilidad)
legacy_out_path = PATHS["data_processed"] / "macro_panel.parquet"
write_parquet_pyarrow(macro_panel_df, legacy_out_path)
print("wrote", legacy_out_path)

# 10) Manifest para abrir facilmente en Data Wrangler
manifest_df = pd.DataFrame(saved_rows).sort_values("dataset").reset_index(drop=True)
manifest_path = STEP_EXPORT_DIR / "00_manifest.parquet"
write_parquet_pyarrow(manifest_df, manifest_path)
manifest_df.to_csv(STEP_EXPORT_DIR / "00_manifest.csv", index=False)
print("[OK] Saved", manifest_path)
print(manifest_df)




EFW columns: ['ISO_Code', 'Countries', 'Year', 'Summary', 'Area 1', 'Area 2', 'Area 3', 'Area 4', 'Area 5', 'Standard Deviation of the 5 EFW Areas', 'World Bank Region', 'World Bank Current Income Classification, 1990-Present']
[OK] Saved /Users/gabrielsaco/Documents/GitHub/economic-freedom/data/processed/macro_pipeline_step_by_step/06_efw_wide.parquet
Merge check (Panel vs EFW): {'left_only': 6930, 'both': 4950, 'right_only': 0}
[OK] Saved /Users/gabrielsaco/Documents/GitHub/economic-freedom/data/processed/macro_pipeline_step_by_step/07_macro_panel.parquet
wrote /Users/gabrielsaco/Documents/GitHub/economic-freedom/data/processed/macro_panel.parquet
[OK] Saved /Users/gabrielsaco/Documents/GitHub/economic-freedom/data/processed/macro_pipeline_step_by_step/00_manifest.parquet
          dataset    rows  cols  \
0        efw_wide    4950    10   
1     macro_panel   11880    74   
2        pwt_wide    9403    45   
3        wdi_long  234360     7   
4  wdi_macro_base   11718    23   
5  wd